# The Evolution of Language Models (2003 - 2017)
This notebook illustrates how neural networks learn language by building three miniature models: **Bengio (2003)**, **Word2Vec (2013)**, and a **Transformer (2017)**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# 1. A tiny toy corpus to learn relationships (genders and royalty)
corpus = [
    "he is a king", 
    "she is a queen",
    "he is a strong man", 
    "she is a wise woman",
    "the king is a man", 
    "the queen is a woman"
]

# 2. Tokenize and build a vocabulary mapping (words -> numbers)
words = " ".join(corpus).split()
vocab = list(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Vocabulary Size: {vocab_size}")
print(f"Vocabulary: {vocab}")

### 1. The Bengio 2003 Architecture (NNLM)
**Key mechanism**: Concatenates word embeddings and passes them through a heavy hidden layer.

In [ ]:
# Rule: Predict the NEXT word from the previous 2 words
bengio_data = []
for sentence in corpus:
    tokens = sentence.split()
    for i in range(2, len(tokens)):
        context = [tokens[i-2], tokens[i-1]]
        target = tokens[i]
        bengio_data.append((context, target))

class Bengio2003(nn.Module):
    def __init__(self, vocab_size, embed_size, context_size, hidden_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        self.linear1 = nn.Linear(context_size * embed_size, hidden_size)
        self.tanh = nn.Tanh()
        self.linear2 = nn.Linear(hidden_size, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs) 
        # CONCATENATE the embeddings
        embeds = embeds.view(embeds.shape[0], -1) 
        hidden = self.tanh(self.linear1(embeds))
        out = self.linear2(hidden)
        return out

model_bengio = Bengio2003(vocab_size, embed_size=10, context_size=2, hidden_size=16)

### 2. The Word2Vec Architecture (CBOW, 2013)
**Key mechanism**: Averages embeddings to create a "Bag of Words". Incredibly fast, but ignores word order completely.

In [ ]:
# Rule: Predict CENTER word from 1 word left, and 1 word right
cbow_data = []
for sentence in corpus:
    tokens = sentence.split()
    for i in range(1, len(tokens)-1):
        context = [tokens[i-1], tokens[i+1]]
        target = tokens[i]
        cbow_data.append((context, target))

class Word2VecCBOW(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        self.linear = nn.Linear(embed_size, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        # AVERAGE the embeddings
        embeds = embeds.mean(dim=1) 
        out = self.linear(embeds)
        return out

model_w2v = Word2VecCBOW(vocab_size, embed_size=10)

### 3. The Transformer Architecture (2017)
**Key mechanism**: Uses **Positional Encoding** (to remember word order) and **Self-Attention** (so words look at each other and dynamically decide which context words are most important). We write the self-attention layer from scratch below so you can see the math in action!

In [ ]:
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, embed_size, context_size):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embed_size)
        self.pos_embeddings = nn.Embedding(context_size, embed_size)
        
        # Self-Attention Matrices (Query, Key, Value)
        self.W_q = nn.Linear(embed_size, embed_size, bias=False)
        self.W_k = nn.Linear(embed_size, embed_size, bias=False)
        self.W_v = nn.Linear(embed_size, embed_size, bias=False)
        
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, inputs, return_attention=False):
        batch_size, seq_len = inputs.shape
        
        # 1. ADD POSITIONS TO WORDS (because Attention math doesn't know word order naturally)
        positions = torch.arange(seq_len, device=inputs.device).unsqueeze(0).expand(batch_size, -1)
        x = self.word_embeddings(inputs) + self.pos_embeddings(positions)
        
        # 2. CREATE QUERIES, KEYS, VALUES
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 3. SELF-ATTENTION (Multiply Q by K to get relevance scores)
        scores = torch.matmul(Q, K.transpose(1, 2)) / (x.size(-1) ** 0.5)
        attention_weights = F.softmax(scores, dim=-1)
        
        # Multiply relevance percentages by V to get final mixed vectors
        context_vectors = torch.matmul(attention_weights, V)
        
        # 4. PREDICT NEXT WORD (Take the vector of the last word in context)
        last_word_vector = context_vectors[:, -1, :]
        out = self.fc(last_word_vector)
        
        if return_attention:
            return out, attention_weights
        return out

model_transformer = MiniTransformer(vocab_size, embed_size=10, context_size=2)

### 4. Train and Visualize the Learned Language

In [ ]:
def train(model, data, epochs=500):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(epochs):
        for context, target in data:
            x = torch.tensor([[word_to_idx[w] for w in context]])
            y = torch.tensor([word_to_idx[target]])
            
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

print("Training Bengio Model...")
train(model_bengio, bengio_data)

print("Training Word2Vec Model...")
train(model_w2v, cbow_data)

print("Training Miniature Transformer...")
train(model_transformer, bengio_data)

def plot_embeddings(model, title, is_transformer=False):
    if is_transformer:
        embeddings = model.word_embeddings.weight.data.numpy()
    else:
        embeddings = model.embeddings.weight.data.numpy()
        
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)
    
    plt.figure(figsize=(7, 4))
    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c='red')
    
    for i, word in enumerate(vocab):
        plt.annotate(word, (embeddings_2d[i, 0]+0.02, embeddings_2d[i, 1]+0.02), fontsize=12)
        
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

plot_embeddings(model_bengio, "Bengio 2003 Word Embeddings")
plot_embeddings(model_w2v, "Word2Vec Word Embeddings")
plot_embeddings(model_transformer, "Transformer Word Embeddings", is_transformer=True)

### 5. Peek Inside the Transformer's Brain
Because we wrote the math out explicitly, we can print the **Attention Matrix** to see which context words the model actually focused on to make its prediction.

In [ ]:
test_phrase = ["the", "king"]
test_idx = torch.tensor([[word_to_idx[w] for w in test_phrase]])

prediction, attention = model_transformer(test_idx, return_attention=True)
predicted_word = idx_to_word[torch.argmax(prediction).item()]

print(f"Input context: '{test_phrase[0]} {test_phrase[1]}'")
print(f"Predicted next word: '{predicted_word}'\n")

# The attention matrix shape is (batch, target_seq, source_seq)
# We want the attention weights used by the LAST word in the context to predict the next word
weights = attention[0, -1, :].detach().numpy()

print("Attention Weights (How much focus the model put on each context word):")
print(f"  '{test_phrase[0]}' -> {weights[0]*100:.1f}%")
print(f"  '{test_phrase[1]}' -> {weights[1]*100:.1f}%")